# Demonstração ao Vivo — AgroSele

Comparação, em tempo real, de três abordagens de seleção de resposta sobre uma
pergunta real de produtor tirada do conjunto de **teste** do MilkQA (nunca
vista por nenhum modelo em treino):

1. **TF-IDF** — léxico puro, sem nenhum modelo de linguagem
2. **Cosseno sobre BERTimbau congelado** — embedding neural, mas sem treino
3. **Cross-Encoder treinado** — self-attention conjunta sobre o par
   pergunta+candidata (ver Seção 4.9 do artigo)

Nada aqui é retreinado ao vivo: todos os caches e checkpoints já estão no
repositório, então cada célula roda em segundos.

**Como usar em sala:** rode a Célula de Setup uma vez (leva ~10-20s), depois
rode as células de cada método em ordem. Para tentar outra pergunta, mude
`INDICE_PERGUNTA` na célula "Passo 1" e rode as células novamente a partir
dali.

## Setup (rodar uma vez)

In [1]:
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset

t0 = time.time()

# --- textos (mesmo export usado pelo cross-encoder) ---
corpus_df = pd.read_csv("../selecao-resposta-milkqa-finetune/datasets/corpus.csv")
queries_df = pd.read_csv("../selecao-resposta-milkqa-finetune/datasets/queries.csv")
texto_resposta = dict(zip(corpus_df["id"].astype(str), corpus_df["text"]))
texto_pergunta = dict(zip(queries_df["id"].astype(str), queries_df["text"]))

# --- split oficial de teste (300 perguntas, nunca usadas em treino) ---
ds = load_dataset("eduagarcia/MilkQA")
conjunto_teste = ds["test"]
print(f"Conjunto de teste: {len(conjunto_teste)} perguntas, 50 candidatas cada")

# --- 1) TF-IDF: indice construido a mao (ver tfidf_model.py) ---
import sys
sys.path.insert(0, "../selecao-resposta-milkqa-classico")
from tfidf_model import TfidfIndex

indice_tfidf = TfidfIndex(list(texto_resposta.keys()), list(texto_resposta.values()))
print(f"Indice TF-IDF construido: {len(indice_tfidf.idf)} termos no vocabulario")

# --- 2) Cosseno sobre BERTimbau congelado: embeddings ja calculados ---
emb_perguntas = torch.load("../selecao-resposta-milkqa/cache/queries_embeddings.pt", weights_only=False)
emb_respostas = torch.load("../selecao-resposta-milkqa/cache/corpus_embeddings.pt", weights_only=False)
print(f"Embeddings BERTimbau congelado: {len(emb_perguntas)} perguntas, {len(emb_respostas)} respostas")

# --- 3) Cross-Encoder treinado: cache de pares + cabeca de classificacao ---
cache_pares = torch.load("../selecao-resposta-milkqa-crossencoder/cache/pair_features_crossencoder.pt", weights_only=False)

class CabecaCrossEncoder(nn.Module):
    def __init__(self, dim_entrada, ocultas, dropout):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(dim_entrada, ocultas), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(ocultas, 1),
        )
    def forward(self, x):
        return self.rede(x).squeeze(-1)

ckpt_ce = torch.load("../selecao-resposta-milkqa-crossencoder/checkpoints/best_model_crossencoder.pt", weights_only=False)
modelo_crossencoder = CabecaCrossEncoder(ckpt_ce["dim_entrada"], ckpt_ce["ocultas"], ckpt_ce["dropout"])
modelo_crossencoder.load_state_dict(ckpt_ce["model_state"])
modelo_crossencoder.eval()
print(f"Cross-Encoder carregado (Accuracy@1 no teste = {ckpt_ce['acuracia1_teste']:.3f})")

print(f"\nSetup completo em {time.time()-t0:.1f}s")

C:\Users\frede\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Conjunto de teste: 300 perguntas, 50 candidatas cada


Indice TF-IDF construido: 13627 termos no vocabulario
Embeddings BERTimbau congelado: 2657 perguntas, 2657 respostas


Cross-Encoder carregado (Accuracy@1 no teste = 0.617)

Setup completo em 7.3s


## Passo 1 — Escolher a pergunta

`INDICE_PERGUNTA` é a posição (0 a 299) dentro do conjunto de teste. Troque o
número e rode a célula de novo para testar outra pergunta ao vivo.

O índice padrão (42) já é um bom exemplo por si só: é uma pergunta sobre
ração de baixo custo em que o Cross-Encoder erra o 1º lugar justamente para
uma resposta quase-gêmea sobre o mesmo assunto — reproduzindo ao vivo o
mesmo padrão de "atrator lexical" discutido na Seção 4.11 do artigo (Análise
de Erros).

In [2]:
INDICE_PERGUNTA = 42  # <-- mude aqui para testar outra pergunta (0 a 299)

linha = conjunto_teste[INDICE_PERGUNTA]
id_pergunta = linha["query-id"]
id_correta = linha["positive-doc-id"]
candidatas = linha["candidates-ids"]

print("PERGUNTA DO PRODUTOR:\n")
print(texto_pergunta[id_pergunta])
print(f"\n({len(candidatas)} respostas candidatas — o gabarito fica escondido até revelarmos os rankings)")

PERGUNTA DO PRODUTOR:

Gostaria muito da atenção da embrapa
como devo apreparar a ração de qualidade com baixo custo para melhorar a
produção do gado leite,
Gostaria de uma receita completa de como devo aprepar a ração, a receita e
os produtos com a quantidade, para produzir uma boa alimentação para o gado
de leite .

(50 respostas candidatas — o gabarito fica escondido até revelarmos os rankings)


## Passo 2 — Ranking por TF-IDF (léxico puro, sem nenhum modelo de linguagem)

In [3]:
t0 = time.time()
vetor_pergunta = indice_tfidf.vectorize_query(texto_pergunta[id_pergunta])
pontuacoes_tfidf = [(cid, TfidfIndex.cosine(vetor_pergunta, indice_tfidf.vectors[cid])) for cid in candidatas]
pontuacoes_tfidf.sort(key=lambda x: -x[1])
ranking_tfidf = [cid for cid, _ in pontuacoes_tfidf]
posicao_tfidf = ranking_tfidf.index(id_correta) + 1

print(f"Tempo: {time.time()-t0:.3f}s\n")
print("Top 5 (TF-IDF):")
for i, (cid, score) in enumerate(pontuacoes_tfidf[:5], start=1):
    marca = "  <-- CORRETA" if cid == id_correta else ""
    print(f"  {i}. score={score:.3f}  {texto_resposta[cid][:90]}...{marca}")
print(f"\nA resposta correta ficou na posição {posicao_tfidf} de {len(candidatas)}.")

Tempo: 0.001s

Top 5 (TF-IDF):
  1. score=0.195  Se a produção do seu gado é de 5 litros por vaca por dia, isso pode ser devido a diferente...
  2. score=0.156  Produção de leite a pasto com suplementação de concentrado.
 
    Os bovinos evoluíram ao ...
  3. score=0.143  Leia o texto abaixo que esclarecerá bastante sobre o uso de volumoso e de concentrado. 
...
  4. score=0.140  Use cana mais uréia na seca.
    Use capim-elefante no período das águas. Neste caso, usa...
  5. score=0.139  De modo geral, a ração concentrada deve ser formulada conforme as necessidades de cada lot...

A resposta correta ficou na posição 11 de 50.


## Passo 3 — Ranking por similaridade de cosseno sobre embeddings do BERTimbau (sem treino)

In [4]:
t0 = time.time()
q = torch.nn.functional.normalize(emb_perguntas[id_pergunta].unsqueeze(0), dim=-1)
a = torch.nn.functional.normalize(torch.stack([emb_respostas[c] for c in candidatas]), dim=-1)
pontuacoes = (a @ q.T).squeeze(-1).numpy()
ordem = np.argsort(-pontuacoes)
ranking_cosseno = [candidatas[i] for i in ordem]
posicao_cosseno = ranking_cosseno.index(id_correta) + 1

print(f"Tempo: {time.time()-t0:.3f}s\n")
print("Top 5 (cosseno, BERTimbau sem treino):")
for i, idx in enumerate(ordem[:5], start=1):
    cid, score = candidatas[idx], pontuacoes[idx]
    marca = "  <-- CORRETA" if cid == id_correta else ""
    print(f"  {i}. score={score:.3f}  {texto_resposta[cid][:90]}...{marca}")
print(f"\nA resposta correta ficou na posição {posicao_cosseno} de {len(candidatas)}.")

Tempo: 0.044s

Top 5 (cosseno, BERTimbau sem treino):
  1. score=0.847  Informamos que è impossível manter os bovinos apenas com trato no cocho e sem uso de volum...
  2. score=0.842  Em relação às suas dúvidas esclarecemos:

  1) Sal proteinado ou mistura multipla: é uma t...
  3. score=0.840  Leia o texto abaixo que esclarecerá bastante sobre o uso de volumoso e de concentrado. 
...
  4. score=0.840  De modo geral, a ração concentrada deve ser formulada conforme as necessidades de cada lot...
  5. score=0.835  Em relação à fórmula e quantidade de ração para gado leiteiro, informamos:

	Para uma co...

A resposta correta ficou na posição 36 de 50.


## Passo 4 — Ranking pelo Cross-Encoder treinado

Diferente dos dois métodos anteriores, aqui pergunta e candidata são
processadas **juntas** pelo BERTimbau (self-attention conjunta sobre a
sequência concatenada `[CLS] pergunta [SEP] candidata [SEP]`), não como dois
embeddings separados.

In [5]:
t0 = time.time()
features = torch.stack([cache_pares[f"{id_pergunta}||{c}"] for c in candidatas])
with torch.no_grad():
    pontuacoes_ce = torch.sigmoid(modelo_crossencoder(features)).numpy()
ordem_ce = np.argsort(-pontuacoes_ce)
ranking_crossencoder = [candidatas[i] for i in ordem_ce]
posicao_crossencoder = ranking_crossencoder.index(id_correta) + 1

print(f"Tempo: {time.time()-t0:.3f}s\n")
print("Top 5 (Cross-Encoder treinado):")
for i, idx in enumerate(ordem_ce[:5], start=1):
    cid, score = candidatas[idx], pontuacoes_ce[idx]
    marca = "  <-- CORRETA" if cid == id_correta else ""
    print(f"  {i}. score={score:.3f}  {texto_resposta[cid][:90]}...{marca}")
print(f"\nA resposta correta ficou na posição {posicao_crossencoder} de {len(candidatas)}.")

Tempo: 0.019s

Top 5 (Cross-Encoder treinado):
  1. score=0.654  Uma ração de boa qualidade e de baixo custo é:
84kg de fubá de milho, 10kg de farelo de so...
  2. score=0.475  Uma ração de boa qualidade e de baixo custo é:

Uma fórmula de ração de baixo custo, espec...  <-- CORRETA
  3. score=0.295  Pode usar uréia agrícola e sulfato de amônio sim, na proporção de 9:1. Para cada 50kg de u...
  4. score=0.284  Use a fórmula como abaixo, sem acrescentar mais nada:   
      Uma outra alternativa de ra...
  5. score=0.254  Sugerimos fornecer um quilo da ração para cada 2,5 litros de leite produzidos acima de 5 l...

A resposta correta ficou na posição 2 de 50.


## Resumo comparativo

In [6]:
print(f"Pergunta #{INDICE_PERGUNTA} (id={id_pergunta})\n")
print(f"{'Método':<32}{'Posição da resposta correta':<30}{'Acertou em 1º?'}")
for nome, posicao in [
    ("TF-IDF (léxico puro)", posicao_tfidf),
    ("Cosseno, BERTimbau sem treino", posicao_cosseno),
    ("Cross-Encoder treinado", posicao_crossencoder),
]:
    acertou = "SIM" if posicao == 1 else "não"
    print(f"{nome:<32}{posicao} de {len(candidatas):<27}{acertou}")

print("\nRESPOSTA CORRETA (gabarito):\n")
print(texto_resposta[id_correta])

Pergunta #42 (id=16475)

Método                          Posição da resposta correta   Acertou em 1º?
TF-IDF (léxico puro)            11 de 50                         não
Cosseno, BERTimbau sem treino   36 de 50                         não
Cross-Encoder treinado          2 de 50                         não

RESPOSTA CORRETA (gabarito):

Uma ração de boa qualidade e de baixo custo é:

Uma fórmula de ração de baixo custo, especialmente para rebanhos de produção média abaixo de 15 litros por vaca por dia, é: 84 kg de fubá de milho; 10 kg de farelo de soja; 2 kg de uréia; 1,5 kg de calcário calcítico; 1,5 kg de sal mineral e 1,0 kg de fosfato bicálcico. Esta ração tem cerca de 19 a 20 % de PB e 70%  de NDT. Fornecer como explicado abaixo.

Caso tenha dificuldades de encontrar os ingredientes pode utilizar uma   fórmula semelhante:
85,0 kg de fubá de milho, 10 kg de farelo de soja, 2 kg de uréia  e 3 kg de sal mineral (ou pode colocar 3 kg de núcleo mineral próprio para formulação de ração 

---

**Quer tentar outra pergunta?** Volte à célula "Passo 1", mude
`INDICE_PERGUNTA` para outro número entre 0 e 299, e rode as células de novo
a partir dali (o Setup não precisa rodar de novo).

Sugestões de perguntas para explorar em sala:
- Alguma em que o TF-IDF vença o Cross-Encoder (mostra que léxico puro ainda
  compete bem em domínio técnico)
- Alguma em que todos os três métodos acertem (mostra convergência quando o
  sinal é forte)
- Peça para a turma escolher um número entre 0 e 299 na hora, sem preparo
  prévio — o método funciona igual em qualquer pergunta do conjunto de teste